# Project FORESIGHT — Demand Forecasting & Risk Scoring (D3 + D4)
Baseline (seasonal-naive) -> Features -> Model -> Backtest -> Risk scoring.
Follows the brief's non-negotiable rule: beat the baseline, honestly,
using rolling-origin backtesting (no future data in features).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
plt.style.use("seaborn-v0_8-whitegrid")

df = pd.read_csv("../data/processed/analysis_ready.csv", parse_dates=["date"])
print(df.shape)

# ---- Aggregate to WEEKLY SKU-level demand (brief asks for weekly forecast) ----
weekly = (
    df.set_index("date")
      .groupby("sku_id")
      .resample("W")["units_sold"]
      .sum()
      .reset_index()
      .rename(columns={"units_sold": "weekly_units"})
)
print(weekly.shape)
weekly.head()

## 1. Frame the metric: WAPE
WAPE = sum(|actual - forecast|) / sum(actual). Robust to low-volume SKUs.


In [ ]:
def wape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.sum(np.abs(y_true))
    if denom == 0:
        return np.nan
    return np.sum(np.abs(y_true - y_pred)) / denom

## 2. Baseline — seasonal-naive
Predicts this week's demand = demand from N weeks ago (season length).
For weekly retail data with limited history, a 1-week-lag naive is the
simplest fair baseline; a longer season length can be tried if history allows.


In [ ]:
SEASON_LAG = 1  # weeks; simplest naive - "same as last week"
HORIZON = 6      # weeks to forecast ahead (per brief: 6-8 week horizon)

weekly = weekly.sort_values(["sku_id", "date"])
weekly["naive_pred"] = weekly.groupby("sku_id")["weekly_units"].shift(SEASON_LAG)

## 3. Feature engineering (lags, rolling stats — no future leakage)


In [ ]:
g = weekly.groupby("sku_id")["weekly_units"]
weekly["lag_1"] = g.shift(1)
weekly["lag_2"] = g.shift(2)
weekly["lag_4"] = g.shift(4)
weekly["roll_mean_4"] = g.shift(1).rolling(4).mean()
weekly["roll_std_4"] = g.shift(1).rolling(4).std()
weekly["week_of_year"] = weekly["date"].dt.isocalendar().week.astype(int)
weekly["month"] = weekly["date"].dt.month

weekly_model = weekly.dropna(subset=["lag_1", "lag_2", "lag_4", "roll_mean_4"]).copy()
print(f"Rows ready for modelling: {weekly_model.shape}")

## 4. Rolling-origin backtest split
Train on the past, test on the most recent HORIZON weeks per SKU —
never a random split for time series.


In [ ]:
cutoff_date = weekly_model["date"].max() - pd.Timedelta(weeks=HORIZON)
train = weekly_model[weekly_model["date"] <= cutoff_date]
test = weekly_model[weekly_model["date"] > cutoff_date]
print(f"Train: {train.shape}, Test: {test.shape}")
print(f"Train up to {cutoff_date.date()}, Test after that")

FEATURES = ["lag_1", "lag_2", "lag_4", "roll_mean_4", "roll_std_4", "week_of_year", "month"]
TARGET = "weekly_units"

X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]

## 5. Baseline WAPE on the test period


In [ ]:
baseline_wape = wape(test[TARGET], test["naive_pred"].fillna(test[TARGET].mean()))
print(f"Baseline (seasonal-naive) WAPE: {baseline_wape:.3f}")

## 6. Train the model
Prefer LightGBM; if unavailable, fall back to sklearn GradientBoostingRegressor
(same idea — gradient-boosted trees — so the workflow still holds).


In [ ]:
try:
    import lightgbm as lgb
    model = lgb.LGBMRegressor(
        n_estimators=300, learning_rate=0.05, max_depth=6, random_state=42
    )
    model.fit(X_train, y_train)
    model_name = "LightGBM"
except ImportError:
    from sklearn.ensemble import GradientBoostingRegressor
    model = GradientBoostingRegressor(
        n_estimators=300, learning_rate=0.05, max_depth=4, random_state=42
    )
    model.fit(X_train, y_train)
    model_name = "GradientBoostingRegressor (sklearn fallback — LightGBM not installed)"

print(f"Trained model: {model_name}")

In [ ]:
preds = model.predict(X_test)
preds = np.clip(preds, 0, None)  # demand can't be negative
model_wape = wape(y_test, preds)

print(f"Baseline WAPE : {baseline_wape:.3f}")
print(f"Model WAPE    : {model_wape:.3f}")
if model_wape < baseline_wape:
    improvement = (1 - model_wape / baseline_wape) * 100
    print(f"✅ Model BEATS baseline by {improvement:.1f}%")
else:
    print("⚠️ Model did NOT beat the baseline on this backtest — "
          "report this honestly; the seasonal-naive baseline should ship instead.")

## 7. Feature importance (explainability)


In [ ]:
importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
plt.figure(figsize=(8, 4))
importances.plot(kind="bar")
plt.title(f"Feature Importance — {model_name}")
plt.ylabel("Importance")
plt.tight_layout()
plt.savefig("../reports/feature_importance.png", dpi=120)
plt.show()
print(importances)

## 8. Illustrative forecast plot for one SKU


In [ ]:
example_sku = test["sku_id"].value_counts().index[0]
ex_train = weekly_model[(weekly_model.sku_id == example_sku) & (weekly_model.date <= cutoff_date)]
ex_test = test[test.sku_id == example_sku].copy()
ex_test["model_pred"] = np.clip(model.predict(ex_test[FEATURES]), 0, None)

plt.figure(figsize=(11, 4))
plt.plot(ex_train["date"], ex_train["weekly_units"], label="Actual (history)", color="black")
plt.plot(ex_test["date"], ex_test["weekly_units"], label="Actual (holdout)", color="black", linestyle="--")
plt.plot(ex_test["date"], ex_test["naive_pred"], label="Seasonal-naive baseline", color="orange")
plt.plot(ex_test["date"], ex_test["model_pred"], label="Model forecast", color="blue")
plt.title(f"Forecast Example — {example_sku}")
plt.xlabel("Week")
plt.ylabel("Units")
plt.legend()
plt.tight_layout()
plt.savefig("../reports/forecast_example.png", dpi=120)
plt.show()

## 9. Produce the forward forecast (next HORIZON weeks per SKU)
For deployment: forecast forward using the latest known lags per SKU.
(Simple iterative approach — refine further if time allows.)


In [ ]:
latest = weekly_model.sort_values("date").groupby("sku_id").tail(1).copy()
future_frames = []
current = latest.copy()

for step in range(1, HORIZON + 1):
    current = current.copy()
    current["date"] = current["date"] + pd.Timedelta(weeks=1)
    current["week_of_year"] = current["date"].dt.isocalendar().week.astype(int)
    current["month"] = current["date"].dt.month
    pred = np.clip(model.predict(current[FEATURES]), 0, None)
    current["forecast"] = pred
    future_frames.append(current[["sku_id", "date", "forecast"]].copy())

    # roll features forward for next iteration
    current["lag_4"] = current["lag_2"]
    current["lag_2"] = current["lag_1"]
    current["lag_1"] = pred
    current["roll_mean_4"] = (current["roll_mean_4"] * 3 + pred) / 4  # rough rolling update

forecast_forward = pd.concat(future_frames, ignore_index=True)
forecast_forward.to_csv("../data/processed/forecast_forward.csv", index=False)
print(f"Forward forecast saved: {forecast_forward.shape}")
forecast_forward.head(10)

## 10. Risk scoring — stockout / overstock (D4)
Combine forecast demand with current inventory position.


In [ ]:
inv = pd.read_csv("../data/inventory_snapshots.csv", parse_dates=["date"])
latest_inv = inv.sort_values("date").groupby("sku_id").tail(1)

# total forecast demand over the horizon, per SKU
horizon_demand = forecast_forward.groupby("sku_id")["forecast"].sum().reset_index()
horizon_demand.columns = ["sku_id", "horizon_demand"]

risk = horizon_demand.merge(latest_inv, on="sku_id", how="left")
risk["available_stock"] = risk["on_hand_units"] + risk["on_order_units"]

# Stockout risk: available stock vs demand over lead time
risk["stockout_risk"] = (risk["horizon_demand"] > risk["available_stock"]).astype(int)
# Overstock risk: on-hand stock far exceeds forecast demand (e.g. >2x)
risk["overstock_risk"] = (risk["on_hand_units"] > 2 * risk["horizon_demand"].clip(lower=1)).astype(int)

def recommend(row):
    if row["stockout_risk"] and not row["overstock_risk"]:
        return "Reorder now"
    if row["overstock_risk"] and not row["stockout_risk"]:
        return "Markdown / clear"
    if row["stockout_risk"] and row["overstock_risk"]:
        return "Watch / volatile"
    return "Healthy"

risk["recommended_action"] = risk.apply(recommend, axis=1)
risk = risk[["sku_id", "horizon_demand", "on_hand_units", "on_order_units",
             "available_stock", "stockout_risk", "overstock_risk", "recommended_action"]]

risk.to_csv("../data/processed/risk_scoring.csv", index=False)
print(risk["recommended_action"].value_counts())
risk.head(15)

## 11. Decisioning grid (visual)


In [ ]:
plt.figure(figsize=(8, 6))
colors = risk["recommended_action"].map({
    "Reorder now": "red", "Markdown / clear": "purple",
    "Watch / volatile": "orange", "Healthy": "green"
})
plt.scatter(risk["overstock_risk"] + np.random.uniform(-0.05, 0.05, len(risk)),
            risk["stockout_risk"] + np.random.uniform(-0.05, 0.05, len(risk)),
            c=colors, s=risk["horizon_demand"].clip(lower=5), alpha=0.6)
plt.xlabel("Overstock risk (jittered)")
plt.ylabel("Stockout risk (jittered)")
plt.title("Decisioning View — Stockout vs Overstock")
plt.tight_layout()
plt.savefig("../reports/decisioning_grid.png", dpi=120)
plt.show()

## 12. Summary for the README / executive readout

Fill in with your actual run's numbers:
- Baseline WAPE: ______
- Model WAPE: ______
- Model beat baseline by ______%  (or: "baseline was retained because the
  model did not beat it — see honest reporting note above")
- SKUs at stockout risk: ______
- SKUs at overstock risk: ______
